# Rerun 2 · Plain-backbone baseline and repeated full model under the ablation protocol (review points 34, 80, and 12/13 support)

**What this fixes.**
* **Point 34:** the architecture-ablation table has no plain backbone (spatial branch only, no morphology branch, no fusion, no attention). This notebook trains it with the identical recipe.
* **Points 80 / 12 / 13:** the full model is trained **again from a fresh initialisation, for each seed in `SEED_LIST`, alongside the ablation arms**, so the ablation deltas can be compared with a baseline from the same procedure and with the run-to-run spread of the same seed.
* Optional (`RUN_REMOVAL_ARMS=True`): re-runs the four removal arms of the architecture table (− spatial, − morphology, − gated fusion, − MS-CA-ECA) under the same loop.

Everything uses the stage-1 recipe (flip-TTA, Mixup, EMA, 50 epochs, seed set per run). Outputs: `plain_backbone_results.csv`, `plain_backbone_predictions.npz`, and McNemar tests of each arm against the freshly trained full model.

**Time.** One run is about the length of the stage-1 training. Default = 2 runs (full model + plain backbone, seed 42). Three seeds or the removal arms multiply that; the results file is appended after every run and finished runs are skipped.

**Attach:** BRISC2025 only.


In [ ]:
import time
SEED_LIST        = [42, 13, 77]   # three seeds
RUN_REMOVAL_ARMS = True           # also runs -Spatial, -Morphology, -GatedFusion, -MS-CA-ECA (6 arms x 3 seeds = 18 runs)
# Seed-42 full model and plain backbone are already done in your earlier session. To skip them: download that session's
# plain_backbone_results.csv and plain_backbone_predictions.npz, upload them as a Kaggle dataset, attach it, and set e.g.
# RESUME_DIR = "/kaggle/input/<your-dataset-name>"   (16 runs remain).  Files are saved after every run, so a stopped session can be resumed the same way.
RESUME_DIR       = None


## §1 · Definitions reused unchanged from the stage-1 notebook (same split, preprocessing, augmentation, loss, Mixup, EMA, flip-TTA, schedule, seed)

Preprocessing is the stage-1 `preprocess()` (percentile clip + min-max stretch, **no AMEF**), the pipeline that produced the reported centralized result.

In [ ]:
!pip install -q -U timm scikit-learn thop scikit-image


In [ ]:
import os, math, random, warnings, hashlib
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                              precision_score, recall_score, roc_auc_score,
                              confusion_matrix, classification_report,
                              matthews_corrcoef, cohen_kappa_score)
import matplotlib.pyplot as plt
import timm
import pandas as pd
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE    = 224
NUM_CLASSES = 4
CLASS_NAMES  = ["glioma", "meningioma", "notumor", "pituitary"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".jfif", ".tif", ".tiff", ".webp"}

OUT_DIR  = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE  = 32
NUM_WORKERS = 2

DROPOUT_RATE  = 0.3
USE_AMP       = True
EARLY_STOP_PATIENCE = 10   # epochs with no val AUC improvement before stopping

print("Device:", DEVICE)
print("Classes:", CLASS_NAMES)
print(f"DROPOUT_RATE={DROPOUT_RATE}  USE_AMP={USE_AMP}")


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def read_gray(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f"Cannot read: {path}")
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY if img.shape[-1] == 3
                           else cv2.COLOR_BGRA2GRAY)
    return img

def robust_normalize(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    p1, p99 = np.percentile(img, [1, 99])
    img = np.clip(img, p1, p99)
    img -= img.min(); img /= (img.max() + 1e-8)
    return img

def brain_mask(img01: np.ndarray) -> np.ndarray:
    """Otsu threshold + morphological cleanup + largest-connected-component
    to isolate brain tissue from black background/border."""
    img_u8 = (np.clip(img01, 0, 1) * 255).astype(np.uint8)
    _, mask = cv2.threshold(img_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num <= 1:
        return np.ones_like(img01, dtype=bool)
    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    brain = (labels == largest)
    if brain.sum() < 0.02 * img01.size:
        return np.ones_like(img01, dtype=bool)
    return brain


def laplacian_noise_estimate(gray_u8: np.ndarray) -> float:
    """Fast noise-sigma estimator (Immerkaer 1996) on an 8-bit grayscale image."""
    H, W = gray_u8.shape
    if H < 3 or W < 3:
        return 0.0
    kernel = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)
    conv = cv2.filter2D(gray_u8.astype(np.float32), -1, kernel)
    sigma = np.sqrt(np.pi / 2) * np.mean(np.abs(conv)) / (6 * (W - 2) * (H - 2) + 1e-8) * (W * H)
    return float(np.clip(sigma, 0.0, 50.0))


def adaptive_gamma(img01: np.ndarray, mask: np.ndarray, target: float = 0.5,
                    max_shift: float = 0.15, deadzone: float = 0.08) -> float:
    """REDESIGNED: gamma that nudges masked-brain mean intensity toward `target`,
    but only when it's meaningfully off (|mu - target| > deadzone) and only
    partway there (capped at max_shift). The original version forced every
    image's mean all the way to exactly 0.5 regardless of starting point,
    which erases inter-image/inter-class intensity variance that
    robust_normalize already preserved -- variance that can carry real
    diagnostic signal (tumor subtypes differ in characteristic T1 intensity).
    This version corrects only genuinely under/over-exposed images and leaves
    normally-exposed ones alone."""
    vals = img01[mask]
    if vals.size == 0:
        return 1.0
    mu = float(np.clip(vals.mean(), 1e-3, 1 - 1e-3))
    if abs(mu - target) <= deadzone:
        return 1.0  # already reasonable -- don't touch it
    bounded_target = mu + np.clip(target - mu, -max_shift, max_shift)
    gamma = np.log(bounded_target) / np.log(mu)
    return float(np.clip(gamma, 0.7, 1.4))  # tighter bound than original [0.5, 2.0]


def masked_entropy(img01: np.ndarray, mask: np.ndarray, bins: int = 32) -> float:
    vals = img01[mask]
    if vals.size == 0:
        return 0.0
    hist, _ = np.histogram(vals, bins=bins, range=(0, 1), density=True)
    hist = hist / (hist.sum() + 1e-8)
    hist = hist[hist > 0]
    return float(-np.sum(hist * np.log2(hist)))


def amef_preprocess(img01: np.ndarray, base_clip: float = 2.0, kappa: float = 0.05,
                     entropy_skip_threshold: float = 3.6) -> np.ndarray:
    """Adaptive MRI Enhancement Framework (v2): noise-gated CLAHE clip limit +
    bounded/deadzoned adaptive gamma correction + entropy-based CLAHE skip.
    Parameter-free at inference time (all thresholds fixed constants).
    Only the gamma step changed vs. v1 -- see adaptive_gamma docstring."""
    mask = brain_mask(img01)
    gray_u8 = (np.clip(img01, 0, 1) * 255).astype(np.uint8)

    sigma_noise = laplacian_noise_estimate(gray_u8)
    gamma = adaptive_gamma(img01, mask)
    img_gamma = np.clip(img01 ** gamma, 0, 1) if gamma != 1.0 else img01

    ent = masked_entropy(img_gamma, mask)
    if ent < entropy_skip_threshold:
        clip_limit = base_clip / (1.0 + kappa * sigma_noise)
        clahe = cv2.createCLAHE(clipLimit=max(clip_limit, 0.5), tileGridSize=(8, 8))
        img_u8 = (img_gamma * 255).astype(np.uint8)
        img_out = clahe.apply(img_u8).astype(np.float32) / 255.0
    else:
        img_out = img_gamma
    return img_out


def to_3ch_tensor(img01: np.ndarray) -> torch.Tensor:
    rgb = np.stack([img01, img01, img01], axis=2)
    rgb = torch.tensor(rgb).permute(2, 0, 1).float()
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (rgb - mean) / std


def preprocess(path: str) -> torch.Tensor:
    """Returns a single [3,224,224] image tensor.
    Pipeline: read_gray -> resize -> robust_normalize -> tensor.
    AMEF (adaptive gamma + entropy-gated CLAHE) is INTENTIONALLY NOT applied
    here. Two independent single-seed ablation runs (original AMEF, then a
    redesigned bounded/deadzoned adaptive_gamma) both showed AMEF losing to
    plain robust-normalize-only preprocessing on every metric (accuracy, F1,
    AUC, ECE) -- see section 21b. `amef_preprocess` and its helpers are kept
    below (unused by this function) only so that ablation cell stays
    reproducible; they are not part of the model's actual preprocessing path."""
    gray = read_gray(path)
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    img01 = robust_normalize(gray)
    return to_3ch_tensor(img01)


# --- Diagnostic utility (call manually, not part of the pipeline) ---
def amef_diagnostic(sample_paths, n=200):
    """Reports how often gamma correction actually fires and by how much,
    plus CLAHE trigger rate. Run before/after any AMEF change to confirm
    the fix is doing what's intended, and to sanity-check against the old
    version's behavior."""
    gammas, mu_before, mu_after, clahe_fired = [], [], [], []
    for p in sample_paths[:n]:
        gray = cv2.resize(read_gray(p), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        img01 = robust_normalize(gray)
        mask = brain_mask(img01)
        mu_before.append(float(img01[mask].mean()))
        gamma = adaptive_gamma(img01, mask)
        gammas.append(gamma)
        img_gamma = np.clip(img01 ** gamma, 0, 1) if gamma != 1.0 else img01
        mu_after.append(float(img_gamma[mask].mean()))
        clahe_fired.append(masked_entropy(img_gamma, mask) < 3.6)

    gammas = np.array(gammas)
    print(f"gamma == 1.0 (untouched): {100*np.mean(gammas == 1.0):.1f}% of images")
    print(f"gamma (when applied):     mean={gammas[gammas!=1.0].mean() if (gammas!=1.0).any() else float('nan'):.3f}")
    print(f"mean-intensity before gamma: {np.mean(mu_before):.3f} ± {np.std(mu_before):.3f}")
    print(f"mean-intensity after  gamma: {np.mean(mu_after):.3f} ± {np.std(mu_after):.3f}")
    print(f"CLAHE fired on {100*np.mean(clahe_fired):.1f}% of images")

print("Preprocessing + AMEF (v2 - bounded/deadzoned gamma) ready.")


In [ ]:
import torchvision.transforms.functional as TF

class MRIAugment:
    def __init__(self, p_flip=0.5, max_rotate=10.0, p_rotate=0.5,
                 brightness=0.2, contrast=0.2, p_jitter=0.5,
                 translate=0.06, p_translate=0.3,
                 use_gamma=True, gamma_range=(0.85, 1.15), p_gamma=0.3):
        self.p_flip = p_flip
        self.max_rotate = max_rotate; self.p_rotate = p_rotate
        self.brightness = brightness; self.contrast = contrast; self.p_jitter = p_jitter
        self.translate = translate; self.p_translate = p_translate
        self.use_gamma = use_gamma; self.gamma_range = gamma_range; self.p_gamma = p_gamma

    def __call__(self, img: torch.Tensor) -> torch.Tensor:
        if random.random() < self.p_flip:
            img = TF.hflip(img)
        if random.random() < self.p_rotate:
            angle = random.uniform(-self.max_rotate, self.max_rotate)
            img = TF.rotate(img, angle)
        if random.random() < self.p_translate:
            tx = random.uniform(-self.translate, self.translate) * IMG_SIZE
            ty = random.uniform(-self.translate, self.translate) * IMG_SIZE
            img = TF.affine(img, angle=0, translate=[int(tx), int(ty)], scale=1.0, shear=0)
        if random.random() < self.p_jitter:
            b = 1.0 + random.uniform(-self.brightness, self.brightness)
            c = 1.0 + random.uniform(-self.contrast, self.contrast)
            img = torch.clamp(img * c + (b - 1.0), -3.0, 3.0)
        if self.use_gamma and random.random() < self.p_gamma:
            gamma = random.uniform(*self.gamma_range)
            img = torch.sign(img) * (torch.abs(img) ** gamma)
        return img

train_augment = MRIAugment()
print("MRIAugment ready.")


In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, samples: List[Dict], is_train: bool = False,
                 augmenter: Optional[MRIAugment] = None):
        self.samples   = samples
        self.is_train  = is_train
        self.augmenter = augmenter

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = preprocess(s["path"])
        if self.is_train and self.augmenter is not None:
            img = self.augmenter(img)
        return {"image": img, "label": torch.tensor(s["label"], dtype=torch.long),
                "class_name": s["class_name"], "path": s["path"]}

print("BrainTumorDataset ready.")


In [ ]:
class ECA(nn.Module):
    """Efficient Channel Attention (Wang et al., 2020). ~5-9 parameters."""
    def __init__(self, channels: int, gamma: int = 2, b: int = 1):
        super().__init__()
        t = int(abs((math.log2(channels) + b) / gamma))
        k = t if t % 2 else t + 1
        k = max(k, 3)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=(k - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y)
        y = self.sigmoid(y).transpose(-1, -2).unsqueeze(-1)
        return x * y.expand_as(x)


class DWSepBlock(nn.Module):
    """Depthwise-separable conv block: DW3x3 -> BN -> act -> PW1x1 -> BN -> act,
    with an optional stride-2 downsample and a residual connection when shapes match."""
    def __init__(self, c_in: int, c_out: int, stride: int = 1, act=nn.SiLU):
        super().__init__()
        self.dw = nn.Conv2d(c_in, c_in, 3, stride=stride, padding=1, groups=c_in, bias=False)
        self.bn1 = nn.BatchNorm2d(c_in)
        self.pw = nn.Conv2d(c_in, c_out, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.act = act(inplace=True)
        self.use_res = (stride == 1 and c_in == c_out)

    def forward(self, x):
        identity = x
        x = self.act(self.bn1(self.dw(x)))
        x = self.bn2(self.pw(x))
        if self.use_res:
            x = x + identity
        return self.act(x)


class SpatialBackbone(nn.Module):
    """Lightweight from-scratch depthwise-separable stem + 3 stages. Exposes
    F1/F2/F3 (stage outputs, strides /4, /8, /16) for multi-scale fusion --
    deliberately NOT a pretrained off-the-shelf backbone."""
    def __init__(self, stem_ch=32, c1=48, c2=88, c3=160, depth=(1, 2, 2)):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, stem_ch, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(stem_ch),
            nn.SiLU(inplace=True),
        )
        self.stage1 = self._make_stage(stem_ch, c1, depth[0])
        self.stage2 = self._make_stage(c1, c2, depth[1])
        self.stage3 = self._make_stage(c2, c3, depth[2])
        self.out_channels = [c1, c2, c3]

    @staticmethod
    def _make_stage(c_in, c_out, n_extra):
        layers = [DWSepBlock(c_in, c_out, stride=2)]
        for _ in range(n_extra):
            layers.append(DWSepBlock(c_out, c_out, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)          # /2
        f1 = self.stage1(x)       # /4
        f2 = self.stage2(f1)      # /8
        f3 = self.stage3(f2)      # /16
        return f1, f2, f3


class MultiScaleSpatial(nn.Module):
    """Spatial representation Fs: projects F1/F2/F3 to a common width, aligns
    to F3's resolution, fuses with learned global softmax weights, then a
    depthwise residual refine."""
    def __init__(self, in_channels_list: List[int], out_channels: int):
        super().__init__()
        self.proj = nn.ModuleList([
            nn.Conv2d(c, out_channels, 1, bias=False) for c in in_channels_list
        ])
        self.bn = nn.ModuleList([nn.BatchNorm2d(out_channels) for _ in in_channels_list])
        self.scale_weight = nn.Parameter(torch.ones(len(in_channels_list)))
        self.refine = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1, groups=out_channels, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, feats: List[torch.Tensor]) -> torch.Tensor:
        target_hw = feats[-1].shape[-2:]
        aligned = [F.adaptive_avg_pool2d(bn(p(f)), target_hw)
                   for p, bn, f in zip(self.proj, self.bn, feats)]
        alpha = torch.softmax(self.scale_weight, dim=0)
        fused = sum(a * x for a, x in zip(alpha, aligned))
        return fused + self.refine(fused)


class MorphologyBranch(nn.Module):
    """Morphology-aware representation Fm. A fixed, parameter-free
    morphological-gradient operator (dilation - erosion via max-pool) applied
    to F3, then a learnable depthwise+pointwise transform, trained end-to-end."""
    def __init__(self, c_in: int, out_channels: int):
        super().__init__()
        self.pool = nn.MaxPool2d(3, stride=1, padding=1)
        self.dw = nn.Conv2d(c_in, c_in, 3, padding=1, groups=c_in, bias=False)
        self.bn1 = nn.BatchNorm2d(c_in)
        self.pw = nn.Conv2d(c_in, out_channels, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        dilation = self.pool(x)
        erosion = -self.pool(-x)
        grad = dilation - erosion          # morphological gradient, parameter-free
        x = self.act(self.bn1(self.dw(grad)))
        x = self.act(self.bn2(self.pw(x)))
        return x


class GatedFusion2(nn.Module):
    """Two-branch per-channel gated fusion (Spatial + Morphology). A
    per-channel, per-sample gate learned from both branches' pooled
    statistics -- gives the network the capacity to weight branches
    differently per output channel and per input image."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        hidden = max(channels // reduction, 8)
        self.gate = nn.Sequential(
            nn.Linear(channels * 2, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels * 2),
        )

    def forward(self, fs: torch.Tensor, fm: torch.Tensor):
        b, c = fs.shape[0], fs.shape[1]
        pooled = torch.cat([self.pool(fs).flatten(1), self.pool(fm).flatten(1)], dim=1)
        raw = self.gate(pooled).view(b, 2, c)
        weights = torch.softmax(raw, dim=1)          # per-channel, per-sample, sums to 1 over the 2 branches
        fused = (weights[:, 0, :].view(b, c, 1, 1) * fs +
                 weights[:, 1, :].view(b, c, 1, 1) * fm)
        return fused, weights


class SimpleAvgFusion(nn.Module):
    """Parameter-free 50/50 averaging fusion. Used ONLY in ablation config D
    ('remove Gated Fusion') so the model still has some way to combine
    Fs/Fm -- not part of the proposed architecture itself."""
    def forward(self, fs: torch.Tensor, fm: torch.Tensor):
        fused = 0.5 * fs + 0.5 * fm
        b, c = fs.shape[0], fs.shape[1]
        weights = torch.full((b, 2, c), 0.5, device=fs.device)
        return fused, weights


class SmallHead(nn.Module):
    """GAP -> small projection -> BatchNorm1d -> GELU -> Dropout -> Linear(4)."""
    def __init__(self, in_dim: int, hidden_dim: int, num_classes: int, dropout: float):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.gap(x).flatten(1)
        emb = self.act(self.bn(self.fc1(x)))
        out = self.fc2(self.drop(emb))
        return out, emb

print("Shared building blocks ready (Spatial branch, Morphology branch, Gated Fusion — unchanged).")


In [ ]:
class MultiScaleContext(nn.Module):
    """Cheap multi-scale contextual extraction: channel-reduce with 1x1, three
    parallel DEPTHWISE 3x3 convs at dilation 1/2/3 (receptive fields 3/5/7),
    concat, expand back with 1x1. No full (non-depthwise, non-1x1) convs, per
    the lightweight-operations requirement."""
    def __init__(self, channels: int, reduction: int = 18):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.reduce = nn.Conv2d(channels, hidden, 1, bias=False)
        self.bn_reduce = nn.BatchNorm2d(hidden)
        self.dw1 = nn.Conv2d(hidden, hidden, 3, padding=1, dilation=1, groups=hidden, bias=False)
        self.dw2 = nn.Conv2d(hidden, hidden, 3, padding=2, dilation=2, groups=hidden, bias=False)
        self.dw3 = nn.Conv2d(hidden, hidden, 3, padding=3, dilation=3, groups=hidden, bias=False)
        self.bn_cat = nn.BatchNorm2d(hidden * 3)
        self.expand = nn.Conv2d(hidden * 3, channels, 1, bias=False)
        self.bn_expand = nn.BatchNorm2d(channels)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        r = self.act(self.bn_reduce(self.reduce(x)))
        m = torch.cat([self.dw1(r), self.dw2(r), self.dw3(r)], dim=1)
        m = self.act(self.bn_cat(m))
        return self.bn_expand(self.expand(m))


class CoordinateAttention(nn.Module):
    """Coordinate Attention (Hou et al., 2021), lightweight form: separately
    pools along H and W to preserve direction-specific information, encodes
    both with a shared 1x1 reduction, then splits back into an H-gate and a
    W-gate (both 1x1 projections, no full convolutions)."""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.reduce = nn.Conv2d(channels, hidden, 1, bias=False)
        self.bn = nn.BatchNorm2d(hidden)
        self.act = nn.SiLU(inplace=True)
        self.conv_h = nn.Conv2d(hidden, channels, 1, bias=False)
        self.conv_w = nn.Conv2d(hidden, channels, 1, bias=False)

    def forward(self, x):
        b, c, h, w = x.shape
        x_h = self.pool_h(x)                          # (b,c,h,1) -- height-direction info
        x_w = self.pool_w(x).permute(0, 1, 3, 2)       # (b,c,w,1) -- width-direction info
        y = torch.cat([x_h, x_w], dim=2)               # (b,c,h+w,1)
        y = self.act(self.bn(self.reduce(y)))
        y_h, y_w = torch.split(y, [h, w], dim=2)
        y_w = y_w.permute(0, 1, 3, 2)                  # back to (b,hidden,1,w)
        g_h = torch.sigmoid(self.conv_h(y_h))          # (b,c,h,1)
        g_w = torch.sigmoid(self.conv_w(y_w))          # (b,c,1,w)
        return x * g_h * g_w


class MSCAECA(nn.Module):
    """Multi-Scale Coordinate-ECA Attention (proposed). Fused feature ->
    parallel {MultiScaleContext, CoordinateAttention} -> ECA channel
    refinement -> residual. use_multiscale / use_coord / use_eca gate each
    stage independently for the internal-ablation study (F-I, section 16);
    a disabled stage is an identity pass-through, so this ONE class serves
    both the full module and every internal-ablation variant -- there is no
    separate ECA/SE/CBAM comparison code."""
    def __init__(self, channels: int, ms_reduction: int = 18, coord_reduction: int = 16,
                 use_multiscale: bool = True, use_coord: bool = True, use_eca: bool = True):
        super().__init__()
        self.use_multiscale = use_multiscale
        self.use_coord = use_coord
        self.use_eca = use_eca
        self.ms = MultiScaleContext(channels, reduction=ms_reduction) if use_multiscale else None
        self.coord = CoordinateAttention(channels, reduction=coord_reduction) if use_coord else None
        self.eca = ECA(channels) if use_eca else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = x
        if self.ms is not None:
            out = out + self.ms(out)
        if self.coord is not None:
            out = self.coord(out)
        if self.eca is not None:
            out = self.eca(out)
        return out + identity

print("MS-CA-ECA (Multi-Scale Coordinate-ECA Attention) ready.")


In [ ]:
# Channel configuration -- rep_width narrowed from SMGF-Net's 160 to 148 to make
# room for MS-CA-ECA under the same 211,560-parameter ceiling (see markdown above).
# Spatial branch / Morphology branch / Gated Fusion configs are otherwise IDENTICAL
# to the reference SMGF-Net.
SMGF_MSCAECA_CONFIG = dict(stem_ch=32, c1=48, c2=88, c3=160, depth=(1, 2, 2),
                            rep_width=148, head_hidden=128)
MSCAECA_CONFIG = dict(ms_reduction=18, coord_reduction=16)
REFERENCE_SMGFNET_PARAMS = 211_560   # previous model (plain ECA), for the required parameter-diff report
PARAM_BUDGET = 211_560               # hard constraint: MS-CA-ECA SMGF-Net <= this


class SMGFNetMSCAECA(nn.Module):
    """SMGF-Net + MS-CA-ECA -- the final proposed architecture, and the ONLY
    model class used throughout this notebook (full model + every ablation
    variant, selected via the flags below).

    Input MRI -> SpatialBackbone (stem + 3 depthwise-separable stages,
    exposing F1/F2/F3) -> {MultiScaleSpatial(Fs), MorphologyBranch(Fm)} ->
    GatedFusion2 -> MS-CA-ECA -> SmallHead -> 4-class logits.

    use_spatial / use_morphology / use_gated_fusion / attention: component
    ablation (section 15, configs A-E). ms_kwargs (use_multiscale /
    use_coord / use_eca): MS-CA-ECA internal ablation (section 16, configs
    F-I). Spatial branch, Morphology branch and Gated Fusion are NOT
    redesigned for any of these ablations -- only removed/replaced wholesale,
    exactly as the task brief specifies.
    """
    def __init__(self, num_classes: int = NUM_CLASSES, dropout: float = DROPOUT_RATE,
                 stem_ch: int = 32, c1: int = 48, c2: int = 88, c3: int = 160,
                 depth=(1, 2, 2), rep_width: int = 148, head_hidden: int = 128,
                 use_spatial: bool = True, use_morphology: bool = True,
                 use_gated_fusion: bool = True, attention: str = "msca_eca",
                 ms_kwargs: Optional[Dict[str, Any]] = None):
        super().__init__()
        assert use_spatial or use_morphology, "at least one of Spatial/Morphology must be active"
        self.use_spatial = use_spatial
        self.use_morphology = use_morphology
        self.use_gated_fusion = use_gated_fusion
        self.attention_type = attention
        ms_kwargs = {**MSCAECA_CONFIG, **(ms_kwargs or {})}

        self.backbone = SpatialBackbone(stem_ch, c1, c2, c3, depth=depth)
        self.multiscale = MultiScaleSpatial(self.backbone.out_channels, rep_width) if use_spatial else None
        self.morphology = MorphologyBranch(c3, rep_width) if use_morphology else None

        if use_spatial and use_morphology:
            self.fusion = GatedFusion2(channels=rep_width) if use_gated_fusion else SimpleAvgFusion()
        else:
            self.fusion = None   # single-branch ablation (B/C): nothing to fuse

        if attention == "msca_eca":
            self.attn = MSCAECA(rep_width, **ms_kwargs)
        elif attention == "eca":
            self.attn = ECA(rep_width)
        elif attention == "none":
            self.attn = nn.Identity()
        else:
            raise ValueError(f"unknown attention type: {attention}")

        self.head = SmallHead(rep_width, head_hidden, num_classes, dropout)

    def forward(self, x: torch.Tensor, return_all: bool = False) -> Dict[str, torch.Tensor]:
        f1, f2, f3 = self.backbone(x)
        fs = self.multiscale([f1, f2, f3]) if self.use_spatial else None
        fm = self.morphology(f3) if self.use_morphology else None

        weights = None
        if self.use_spatial and self.use_morphology:
            fused, weights = self.fusion(fs, fm)
        elif self.use_spatial:
            fused = fs
        else:
            fused = fm

        attn = self.attn(fused)
        logits, emb = self.head(attn)
        out = {"logits": logits, "emb": emb}
        if return_all:
            out.update({"f1": f1, "f2": f2, "f3": f3, "spatial": fs, "morphology": fm,
                        "fusion_weights": weights, "fused": fused, "attn": attn})
        return out


def build_smgf_mscaeca(**kwargs) -> SMGFNetMSCAECA:
    """Builds SMGF-Net + MS-CA-ECA (or an ablation variant of it). Any of
    SMGFNetMSCAECA's constructor args can be overridden via kwargs, e.g.
    build_smgf_mscaeca(use_spatial=False) or
    build_smgf_mscaeca(ms_kwargs=dict(use_coord=False))."""
    cfg = {**SMGF_MSCAECA_CONFIG}
    ms_override = kwargs.pop("ms_kwargs", None)
    cfg.update(kwargs)
    if ms_override is not None:
        cfg["ms_kwargs"] = ms_override
    model = SMGFNetMSCAECA(**cfg).to(DEVICE)
    return model


def param_breakdown(model: "SMGFNetMSCAECA") -> Dict[str, int]:
    def count(m):
        return 0 if m is None else sum(p.numel() for p in m.parameters() if p.requires_grad)
    parts = {
        "backbone (stem + 3 DW-sep stages)": count(model.backbone),
        "spatial branch (F1/F2/F3 -> Fs)": count(model.multiscale),
        "morphology branch (Fm)": count(model.morphology),
        "gated fusion (Fs, Fm -> fused)": count(model.fusion),
        "MS-CA-ECA attention": count(model.attn),
        "classifier head": count(model.head),
    }
    parts["TOTAL"] = sum(v for k, v in parts.items())
    return parts


print("build_smgf_mscaeca() ready.")


In [ ]:
def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15):
    """Returns (ece, bin_accs, bin_confs, bin_counts)."""
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(np.float32)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_accs, bin_confs, bin_counts = [], [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        count = mask.sum()
        if count == 0:
            bin_accs.append(0); bin_confs.append(0); bin_counts.append(0)
            continue
        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += (count / len(labels)) * abs(bin_acc - bin_conf)
        bin_accs.append(bin_acc); bin_confs.append(bin_conf); bin_counts.append(count)
    return ece, bin_accs, bin_confs, bin_counts


def compute_class_weights(samples: List[Dict], num_classes: int = NUM_CLASSES) -> torch.Tensor:
    """Inverse-frequency class weights, normalized to mean 1.0."""
    counts = np.zeros(num_classes)
    for s in samples:
        counts[s["label"]] += 1
    counts = np.maximum(counts, 1)
    weights = counts.sum() / (num_classes * counts)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


class CAWLPlus(nn.Module):
    def __init__(self, class_weights: torch.Tensor, num_classes: int = NUM_CLASSES,
                 emb_dim: int = 64, label_smoothing: float = 0.05,
                 target_confidence_cap: float = 0.85, lambda_max: float = 0.3,
                 anneal_epochs: int = 15, proto_momentum: float = 0.9,
                 proto_margin: float = 0.5, beta: float = 0.1,
                 proto_warmup_epochs: int = 8):
        super().__init__()
        self.register_buffer("class_weights", class_weights)
        self.label_smoothing = label_smoothing
        self.target_confidence_cap = target_confidence_cap
        self.lambda_max = lambda_max
        self.anneal_epochs = anneal_epochs
        self.current_epoch = 0

        self.num_classes = num_classes
        self.proto_momentum = proto_momentum
        self.proto_margin = proto_margin
        self.beta = beta
        self.proto_warmup_epochs = proto_warmup_epochs
        self.register_buffer("prototypes", torch.zeros(num_classes, emb_dim))
        self.register_buffer("proto_initialized", torch.zeros(num_classes, dtype=torch.bool))

    def _lambda_t(self):
        frac = min(self.current_epoch / max(self.anneal_epochs, 1), 1.0)
        return self.lambda_max * frac

    @torch.no_grad()
    def _update_prototypes(self, emb: torch.Tensor, targets: torch.Tensor):
        for c in range(self.num_classes):
            mask = targets == c
            if mask.sum() == 0:
                continue
            batch_mean = emb[mask].mean(dim=0)
            if not self.proto_initialized[c]:
                self.prototypes[c] = batch_mean
                self.proto_initialized[c] = True
            else:
                self.prototypes[c] = (self.proto_momentum * self.prototypes[c] +
                                       (1 - self.proto_momentum) * batch_mean)

    def _prototype_margin_loss(self, emb: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if not self.proto_initialized.all():
            return torch.tensor(0.0, device=emb.device)
        protos = self.prototypes.to(emb.device)
        d = torch.cdist(emb, protos) ** 2
        d_pos = d.gather(1, targets.view(-1, 1)).squeeze(1)
        d_masked = d.clone()
        d_masked.scatter_(1, targets.view(-1, 1), float("inf"))
        d_neg = d_masked.min(dim=1).values
        return F.relu(d_pos - d_neg + self.proto_margin).mean()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor,
                emb: Optional[torch.Tensor] = None) -> torch.Tensor:
        ce = F.cross_entropy(logits, targets, weight=self.class_weights,
                              label_smoothing=self.label_smoothing)
        probs = F.softmax(logits, dim=1)
        conf, _ = probs.max(dim=1)
        pred = logits.argmax(dim=1)
        correct = (pred == targets).float()
        overconf = F.relu(conf - self.target_confidence_cap) * correct
        conf_penalty = overconf.mean()
        loss = ce + self._lambda_t() * conf_penalty

        if emb is not None and self.training:
            self._update_prototypes(emb.detach(), targets)   # EMA warms up silently from epoch 1
        if emb is not None and self.current_epoch >= self.proto_warmup_epochs:
            loss = loss + self.beta * self._prototype_margin_loss(emb, targets)  # gated
        return loss

print("CAWLPlus ready (with proto_warmup_epochs gate).")


In [ ]:
def _file_hash(path: str) -> str:
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def dedupe_splits(splits: Dict[str, List[Dict]],
                   priority: List[str] = ["test", "val", "train"]) -> Dict[str, List[Dict]]:
    hash_to_owner: Dict[str, Tuple[str, int]] = {}
    to_remove: Dict[str, set] = {name: set() for name in splits}
    n_cross = n_intra = 0

    for name in sorted(splits, key=lambda n: priority.index(n) if n in priority else 99):
        for idx, s in enumerate(splits[name]):
            h = _file_hash(s["path"])
            if h in hash_to_owner:
                owner_name, owner_idx = hash_to_owner[h]
                if owner_name == name:
                    n_intra += 1
                    if n_intra <= 5:
                        print(f"  [DUPLICATE within '{name}'] {splits[name][owner_idx]['path']}  ==  {s['path']}")
                else:
                    n_cross += 1
                    if n_cross <= 5:
                        print(f"  [LEAKAGE: kept in '{owner_name}', removed from '{name}'] {s['path']}")
                to_remove[name].add(idx)
            else:
                hash_to_owner[h] = (name, idx)

    cleaned = {name: [s for i, s in enumerate(samples) if i not in to_remove[name]]
               for name, samples in splits.items()}
    print(f"[LEAKAGE CHECK] {n_cross} cross-split leak(s) + {n_intra} intra-split "
          f"duplicate(s) auto-removed ({n_cross + n_intra} total).")
    for name in splits:
        removed = len(to_remove[name])
        if removed:
            print(f"  {name}: {len(splits[name])} -> {len(cleaned[name])} ({removed} removed)")
    return cleaned


def scan_dataset_generic(root: Path, folder_to_class: Dict[str, str], split_name: str) -> List[Dict[str, Any]]:
    samples = []
    for folder_name, cls in folder_to_class.items():
        cls_dir = root / folder_name
        if not cls_dir.exists():
            print(f"  [WARN] missing: {cls_dir}")
            continue
        n_before = len(samples)
        for p in cls_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                samples.append({"path": str(p), "label": CLASS_TO_IDX[cls],
                                 "class_name": cls, "split": split_name})
        print(f"  {split_name}/{folder_name} -> {cls}: {len(samples) - n_before} images")
    print(f"{split_name}: {len(samples)} images across {NUM_CLASSES} classes")
    return samples

print("Scanner + leakage-check utilities ready.")


In [ ]:
BRISC_ROOT = Path("/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task")
BRISC_TRAIN_ROOT = BRISC_ROOT / "train"
BRISC_TEST_ROOT  = BRISC_ROOT / "test"
print(f"BRISC2025 train exists: {BRISC_TRAIN_ROOT.exists()} -> {BRISC_TRAIN_ROOT}")
print(f"BRISC2025 test  exists: {BRISC_TEST_ROOT.exists()} -> {BRISC_TEST_ROOT}")
if not (BRISC_TRAIN_ROOT.exists() and BRISC_TEST_ROOT.exists()):
    raise RuntimeError("BRISC2025 not found. Make sure it's attached to this "
                        "Kaggle notebook (Add Input -> search 'brisc2025').")

BRISC_FOLDER_TO_CLASS = {
    "glioma": "glioma", "meningioma": "meningioma",
    "no_tumor": "notumor", "pituitary": "pituitary",
}


In [ ]:
TOTAL_EPOCHS  = 50
WEIGHT_DECAY  = 2e-4
GRAD_CLIP     = 1.0
PEAK_LR       = 1.8e-4   # was 4e-4 -- too high a peak was a major source of curve noise
MIN_LR        = 1e-6

WARMUP_EPOCHS_V2     = 15   # was 10 -- longer, gentler ramp instead of a hard 10-epoch climb
WARMUP_START_DIV_V2  = 50

FREEZE_BACKBONE_EPOCHS = 3

# Fine-grained early logging: for the first FINEGRAIN_EPOCHS epochs, evaluate
# train(clean)/val this many times DURING the epoch (not just once at the end).
# A frozen-backbone linear-probe head converges fast -- within a single epoch it can
# already climb from chance-level (~25% for 4 classes) most of the way to its plateau.
# Logging only once per epoch was hiding that climb and making it look like val
# "jumped" straight to a high number with no visible ramp. This makes the ramp visible.
FINEGRAIN_EPOCHS   = 2
FINEGRAIN_SUBSTEPS = 5

def lr_at_epoch_v2(epoch: int) -> float:
    warmup_start_lr = PEAK_LR / WARMUP_START_DIV_V2
    if epoch <= WARMUP_EPOCHS_V2:
        frac = epoch / WARMUP_EPOCHS_V2
        return warmup_start_lr + frac * (PEAK_LR - warmup_start_lr)
    remaining_epochs = TOTAL_EPOCHS - WARMUP_EPOCHS_V2
    progress = min((epoch - WARMUP_EPOCHS_V2) / max(remaining_epochs, 1), 1.0)
    return MIN_LR + 0.5 * (PEAK_LR - MIN_LR) * (1 + math.cos(math.pi * progress))


def train_one_epoch_v2(model, loader, optimizer, criterion: CAWLPlus,
                        scaler: Optional[GradScaler] = None):
    model.train()
    total_loss = correct = total = 0
    use_amp = scaler is not None
    for batch in loader:
        img   = batch["image"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with autocast():
                out  = model(img)
                loss = criterion(out["logits"], label, out["emb"])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(img)
            loss = criterion(out["logits"], label, out["emb"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        pred = out["logits"].argmax(1)
        correct += (pred == label).sum().item()
        total   += label.size(0)
        total_loss += loss.item() * label.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_v2(model, loader, criterion: CAWLPlus, device: str = DEVICE) -> Dict[str, Any]:
    model.eval()
    all_preds, all_labels, all_probs, total_loss = [], [], [], 0.0
    for batch in loader:
        img   = batch["image"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)
        out   = model(img)
        loss  = criterion(out["logits"], label, out["emb"])
        total_loss += loss.item() * label.size(0)
        probs = torch.softmax(out["logits"], dim=1).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_preds.extend(out["logits"].argmax(1).cpu().tolist())
        all_labels.extend(label.cpu().tolist())

    n = len(all_labels)
    y_true = np.array(all_labels); y_pred = np.array(all_preds); y_prob = np.array(all_probs)
    try:
        auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auc = float("nan")
    cm = confusion_matrix(y_true, y_pred)
    ece, _, _, _ = expected_calibration_error(y_prob, y_true)

    return {
        "loss": total_loss / n,
        "acc": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "precision_mac": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_mac": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_mac": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "auc": auc,
        "mcc": matthews_corrcoef(y_true, y_pred),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "ece": ece,
        "cm": cm,
        "report": classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0),
    }


def plot_confusion_matrix(cm: np.ndarray, title: str = "Confusion Matrix", save_path: Optional[str] = None):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.colorbar(im, ax=ax); plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

# Fine-grained ("chunked") epoch trainer -- identical math to train_one_epoch_v2,
# but pauses n_chunks-1 times during the epoch to log a clean train/val snapshot at
# that fractional-epoch point. Only used for the first FINEGRAIN_EPOCHS epochs.
def train_one_epoch_chunked_v2(model, loader, optimizer, criterion: CAWLPlus,
                                scaler: Optional[GradScaler], n_chunks: int, epoch: int,
                                train_eval_loader, val_loader, history: Dict[str, list]):
    model.train()
    use_amp = scaler is not None
    batches = list(loader)
    n_batches = len(batches)
    chunk_size = max(1, n_batches // n_chunks)
    total_loss = correct = total = 0

    for i, batch in enumerate(batches):
        img   = batch["image"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with autocast():
                out  = model(img)
                loss = criterion(out["logits"], label, out["emb"])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(img)
            loss = criterion(out["logits"], label, out["emb"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        pred = out["logits"].argmax(1)
        correct += (pred == label).sum().item()
        total   += label.size(0)
        total_loss += loss.item() * label.size(0)

        is_chunk_boundary = ((i + 1) % chunk_size == 0) and (i + 1) < n_batches
        if is_chunk_boundary:
            frac_epoch = (epoch - 1) + (i + 1) / n_batches
            train_clean = evaluate_v2(model, train_eval_loader, criterion)
            val_metrics = evaluate_v2(model, val_loader, criterion)
            print(f"  epoch {frac_epoch:06.2f} (substep)  "
                  f"train_acc(clean)={train_clean['acc']:.4f}  "
                  f"val_acc={val_metrics['acc']:.4f}  val_auc={val_metrics['auc']:.4f}")
            history["epoch"].append(frac_epoch)
            history["train_loss"].append(train_clean["loss"])
            history["train_acc"].append(train_clean["acc"])
            history["train_loss_aug"].append(total_loss / total)
            history["train_acc_aug"].append(correct / total)
            history["val_loss"].append(val_metrics["loss"])
            history["val_acc"].append(val_metrics["acc"])
            history["val_auc"].append(val_metrics["auc"])
            history["val_ece"].append(val_metrics["ece"])
            history["lr"].append(optimizer.param_groups[0]["lr"])
            model.train()

    return total_loss / total, correct / total


print(f"Schedule ready: {TOTAL_EPOCHS} epochs, {WARMUP_EPOCHS_V2}-epoch gentle warmup "
      f"(start={PEAK_LR/WARMUP_START_DIV_V2:.1e}), peak_lr={PEAK_LR:.1e}, AMP={USE_AMP}, "
      f"early_stop_patience={EARLY_STOP_PATIENCE}, freeze_backbone_epochs={FREEZE_BACKBONE_EPOCHS}, "
      f"finegrain_epochs={FINEGRAIN_EPOCHS}x{FINEGRAIN_SUBSTEPS}")


In [ ]:
class ModelEMA:
    """Exponential moving average of the model's weights."""
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


def mixup_batch(img: torch.Tensor, label: torch.Tensor, alpha: float = 0.2):
    """Standard Mixup (Zhang et al., 2018)."""
    if alpha <= 0:
        return img, label, label, 1.0
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(img.size(0), device=img.device)
    mixed = lam * img + (1 - lam) * img[idx]
    return mixed, label, label[idx], lam


def mixup_cawl_loss(criterion: CAWLPlus, logits: torch.Tensor, emb: torch.Tensor,
                     y_a: torch.Tensor, y_b: torch.Tensor, lam: float) -> torch.Tensor:
    """Mixup-safe re-implementation of CAWLPlus.forward() (§9): computes CE +
    confidence-penalty for both mixed labels, but touches the prototype
    buffer only once per batch (dominant label), to avoid the in-place
    prototype-mutation issue of calling criterion(...) twice."""
    ce_a = F.cross_entropy(logits, y_a, weight=criterion.class_weights,
                            label_smoothing=criterion.label_smoothing)
    ce_b = F.cross_entropy(logits, y_b, weight=criterion.class_weights,
                            label_smoothing=criterion.label_smoothing)
    ce = lam * ce_a + (1 - lam) * ce_b

    probs = F.softmax(logits, dim=1)
    conf, _ = probs.max(dim=1)
    pred = logits.argmax(dim=1)
    correct = lam * (pred == y_a).float() + (1 - lam) * (pred == y_b).float()
    conf_penalty = (F.relu(conf - criterion.target_confidence_cap) * correct).mean()
    loss = ce + criterion._lambda_t() * conf_penalty

    y_dom = y_a if lam >= 0.5 else y_b
    if criterion.training:
        criterion._update_prototypes(emb.detach(), y_dom)
    if criterion.current_epoch >= criterion.proto_warmup_epochs:
        loss = loss + criterion.beta * criterion._prototype_margin_loss(emb, y_dom)
    return loss


def train_one_epoch_mixup(model, loader, optimizer, criterion: CAWLPlus,
                           scaler: Optional[GradScaler] = None,
                           ema: Optional[ModelEMA] = None,
                           mixup_alpha: float = 0.2):
    model.train()
    total_loss = correct = total = 0
    use_amp = scaler is not None
    for batch in loader:
        img   = batch["image"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        img_m, y_a, y_b, lam = mixup_batch(img, label, alpha=mixup_alpha)
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with autocast():
                out  = model(img_m)
                loss = mixup_cawl_loss(criterion, out["logits"], out["emb"], y_a, y_b, lam)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(img_m)
            loss = mixup_cawl_loss(criterion, out["logits"], out["emb"], y_a, y_b, lam)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        if ema is not None:
            ema.update(model)

        pred = out["logits"].argmax(1)
        correct += (lam * (pred == y_a).float() + (1 - lam) * (pred == y_b).float()).sum().item()
        total   += label.size(0)
        total_loss += loss.item() * label.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_v2_tta(model, loader, criterion: CAWLPlus, device: str = DEVICE) -> Dict[str, Any]:
    """Same metric suite as evaluate_v2 (§12), but averages softmax
    probabilities over the original image and its horizontal flip
    (test-time augmentation) before taking the argmax."""
    model.eval()
    all_preds, all_labels, all_probs, total_loss = [], [], [], 0.0
    for batch in loader:
        img   = batch["image"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)
        out1  = model(img)
        out2  = model(torch.flip(img, dims=[3]))
        probs = (torch.softmax(out1["logits"], dim=1) + torch.softmax(out2["logits"], dim=1)) / 2
        loss  = criterion(out1["logits"], label, out1["emb"])
        total_loss += loss.item() * label.size(0)
        all_probs.extend(probs.cpu().numpy().tolist())
        all_preds.extend(probs.argmax(1).cpu().tolist())
        all_labels.extend(label.cpu().tolist())

    n = len(all_labels)
    y_true = np.array(all_labels); y_pred = np.array(all_preds); y_prob = np.array(all_probs)
    try:
        auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auc = float("nan")
    cm = confusion_matrix(y_true, y_pred)
    ece, _, _, _ = expected_calibration_error(y_prob, y_true)
    return {
        "loss": total_loss / n,
        "acc": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "precision_mac": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_mac": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_mac": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "auc": auc,
        "mcc": matthews_corrcoef(y_true, y_pred),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "ece": ece,
        "cm": cm,
        "report": classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0),
    }


def run_experiment(dataset_name: str, train_root: Path, test_root: Path,
                    folder_to_class: Dict[str, str],
                    model_builder=None,
                    total_epochs: Optional[int] = None,
                    dropout: Optional[float] = None,
                    weight_decay: Optional[float] = None,
                    mixup_alpha: float = 0.2,
                    ema_decay: float = 0.999,
                    model_kwargs: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Trains a model built by `model_builder(dropout=..., **model_kwargs)`
    (defaults to build_smgf_mscaeca -- the full SMGF-Net + MS-CA-ECA model)
    with the SAME dataset loading / splitting / leakage-check / preprocessing
    / augmentation / optimizer / LR-schedule / CAWL+ loss / Mixup / EMA /
    flip-TTA pipeline every time. Used for the full model (§14) AND every
    component/internal ablation variant (§15/§16) -- this identical function
    call is what keeps the ablation protocol fair (section 5 of the task
    brief): same splits, same preprocessing, same augmentation, same image
    size, same batch size, same optimizer, same LR schedule, same epochs,
    same loss, same early stopping, same seed."""
    model_builder = model_builder or build_smgf_mscaeca
    total_epochs = TOTAL_EPOCHS if total_epochs is None else total_epochs
    dropout = 0.4 if dropout is None else dropout
    weight_decay = WEIGHT_DECAY * 1.5 if weight_decay is None else weight_decay
    model_kwargs = model_kwargs or {}

    print("\n" + "#" * 65)
    print(f"# EXPERIMENT: {dataset_name}  "
          f"[dropout={dropout} weight_decay={weight_decay:.2e} mixup_alpha={mixup_alpha} "
          f"ema_decay={ema_decay} epochs={total_epochs} model_kwargs={model_kwargs}]")
    print("#" * 65)

    all_train = scan_dataset_generic(train_root, folder_to_class, "train")
    test_s    = scan_dataset_generic(test_root, folder_to_class, "test")
    if len(all_train) == 0 or len(test_s) == 0:
        raise RuntimeError(f"[{dataset_name}] No images found — check paths.")
    train_s, val_s = train_test_split(
        all_train, test_size=0.15, random_state=SEED,
        stratify=[s["label"] for s in all_train])

    splits = dedupe_splits({"train": train_s, "val": val_s, "test": test_s})
    train_s, val_s, test_s = splits["train"], splits["val"], splits["test"]

    train_ds = BrainTumorDataset(train_s, is_train=True, augmenter=train_augment)
    val_ds   = BrainTumorDataset(val_s,   is_train=False)
    test_ds  = BrainTumorDataset(test_s,  is_train=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)

    if len(train_s) > len(val_s):
        train_eval_s, _ = train_test_split(
            train_s, train_size=len(val_s), random_state=SEED,
            stratify=[s["label"] for s in train_s])
    else:
        train_eval_s = train_s
    train_eval_ds = BrainTumorDataset(train_eval_s, is_train=False)
    train_eval_loader = DataLoader(train_eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True)

    model = model_builder(dropout=dropout, **model_kwargs)
    total_params = sum(p.numel() for p in model.parameters())
    ema = ModelEMA(model, decay=ema_decay)

    class_weights = compute_class_weights(train_s).to(DEVICE)
    criterion = CAWLPlus(class_weights, num_classes=NUM_CLASSES,
                          emb_dim=model.head.fc1.out_features).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, weight_decay=weight_decay)
    scaler = GradScaler() if USE_AMP else None

    history = {"epoch": [], "train_loss": [], "train_acc": [], "train_loss_aug": [], "train_acc_aug": [],
               "val_loss": [], "val_acc": [], "val_auc": [], "val_ece": [], "lr": []}

    _t0 = evaluate_v2(model, train_eval_loader, criterion)
    _v0 = evaluate_v2(model, val_loader, criterion)
    print(f"  epoch 000 (pre-training baseline)  "
          f"train_acc(clean)={_t0['acc']:.4f}  val_acc={_v0['acc']:.4f}  val_auc={_v0['auc']:.4f}")
    history["epoch"].append(0)
    history["train_loss"].append(_t0["loss"]); history["train_acc"].append(_t0["acc"])
    history["train_loss_aug"].append(_t0["loss"]); history["train_acc_aug"].append(_t0["acc"])
    history["val_loss"].append(_v0["loss"]); history["val_acc"].append(_v0["acc"])
    history["val_auc"].append(_v0["auc"]); history["val_ece"].append(_v0["ece"])
    history["lr"].append(0.0)

    best_val_auc = -1.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, total_epochs + 1):
        for g in optimizer.param_groups:
            g["lr"] = lr_at_epoch_v2(epoch)
        criterion.current_epoch = epoch

        train_loss, train_acc = train_one_epoch_mixup(
            model, train_loader, optimizer, criterion, scaler, ema=ema, mixup_alpha=mixup_alpha)
        train_clean = evaluate_v2(model, train_eval_loader, criterion)
        val_metrics = evaluate_v2(model, val_loader, criterion)

        print(f"  epoch {epoch:03d}  lr={lr_at_epoch_v2(epoch):.2e}  "
              f"train_acc(clean)={train_clean['acc']:.4f} train_acc(mixup)={train_acc:.4f}  "
              f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['acc']:.4f} "
              f"val_auc={val_metrics['auc']:.4f} val_ece={val_metrics['ece']:.4f}")

        history["epoch"].append(epoch)
        history["train_loss"].append(train_clean["loss"]); history["train_acc"].append(train_clean["acc"])
        history["train_loss_aug"].append(train_loss); history["train_acc_aug"].append(train_acc)
        history["val_loss"].append(val_metrics["loss"]); history["val_acc"].append(val_metrics["acc"])
        history["val_auc"].append(val_metrics["auc"]); history["val_ece"].append(val_metrics["ece"])
        history["lr"].append(lr_at_epoch_v2(epoch))

        if val_metrics["auc"] > best_val_auc:
            best_val_auc = val_metrics["auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f"  [EARLY STOP] no val AUC improvement for {EARLY_STOP_PATIENCE} epochs.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    raw_val = evaluate_v2(model, val_loader, criterion)

    ema_model = model_builder(dropout=dropout, **model_kwargs)
    ema.copy_to(ema_model)
    ema_val = evaluate_v2(ema_model, val_loader, criterion)

    if ema_val["auc"] >= raw_val["auc"]:
        print(f"  [FINAL WEIGHTS] EMA (val_auc={ema_val['auc']:.4f}) >= "
              f"best-checkpoint (val_auc={raw_val['auc']:.4f}) -> using EMA weights.")
        model = ema_model
    else:
        print(f"  [FINAL WEIGHTS] best-checkpoint (val_auc={raw_val['auc']:.4f}) > "
              f"EMA (val_auc={ema_val['auc']:.4f}) -> keeping best-checkpoint weights.")
        del ema_model

    test_metrics = evaluate_v2_tta(model, test_loader, criterion)
    print(f"[{dataset_name} [TEST, flip-TTA]] Accuracy={test_metrics['acc']:.4f}  "
          f"BalancedAcc={test_metrics['balanced_acc']:.4f}  F1(macro)={test_metrics['f1_mac']:.4f}  "
          f"AUC={test_metrics['auc']:.4f}  MCC={test_metrics['mcc']:.4f}  "
          f"CohenKappa={test_metrics['kappa']:.4f}  ECE={test_metrics['ece']:.4f}")
    print(test_metrics["report"])
    plot_confusion_matrix(test_metrics["cm"], title=f"{dataset_name} Confusion Matrix (flip-TTA)",
                           save_path=str(OUT_DIR / f"{dataset_name}_confusion_matrix.png"))

    torch.save(model.state_dict(), OUT_DIR / f"{dataset_name}.pth")

    return {
        "name": dataset_name, "model": model, "criterion": criterion,
        "total_params": total_params,
        "n_train": len(train_s), "n_val": len(val_s), "n_test": len(test_s),
        "train_samples": train_s, "val_samples": val_s, "test_samples": test_s,
        "test_metrics": test_metrics,
        "history": history,
        "config": {"dropout": dropout, "weight_decay": weight_decay, "mixup_alpha": mixup_alpha,
                   "ema_decay": ema_decay, "epochs": total_epochs, **SMGF_MSCAECA_CONFIG, **model_kwargs},
    }

print("run_experiment() ready (Mixup + EMA + flip-TTA, model-builder-generic).")


## §2 · Arms and runner

In [ ]:

# ---- helpers: paired test-set predictions with flip-TTA (not returned by run_experiment) ----
@torch.no_grad()
def predict_tta(model, samples):
    ds = BrainTumorDataset(samples, is_train=False)
    dl = DataLoader(ds, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
    model.eval(); P1=[]; P2=[]; Y=[]
    for b in dl:
        x=b["image"].to(DEVICE); y=b["label"]
        p1=torch.softmax(model(x)["logits"],1); p2=torch.softmax(model(torch.flip(x,dims=[3]))["logits"],1)
        P1.append(p1.cpu().numpy()); P2.append(p2.cpu().numpy()); Y.append(y.numpy())
    P1=np.concatenate(P1); P2=np.concatenate(P2)
    return np.concatenate(Y), P1, (P1+P2)/2

def mcnemar_exact(correct_a, correct_b):
    """Exact two-sided McNemar test on paired correctness vectors. Returns (b, c, p): b = A right/B wrong, c = A wrong/B right."""
    from scipy.stats import binomtest
    b=int(np.sum(correct_a & ~correct_b)); c=int(np.sum(~correct_a & correct_b))
    n=b+c
    p=1.0 if n==0 else binomtest(min(b,c), n, 0.5, alternative="two-sided").pvalue
    return b,c,float(p)

def paired_bootstrap_diff(correct_a, correct_b, n_boot=2000, seed=0):
    rng=np.random.default_rng(seed); n=len(correct_a); d=[]
    for _ in range(n_boot):
        idx=rng.integers(0,n,n); d.append(correct_a[idx].mean()-correct_b[idx].mean())
    return float(np.mean(d)), float(np.percentile(d,2.5)), float(np.percentile(d,97.5))

def count_params(m, exclude=()):
    return sum(p.numel() for n,p in m.named_parameters() if not any(n.startswith(e) for e in exclude))

def efficiency(model, n_iter=200, warm=30):
    from thop import profile as thop_profile
    import copy
    x=torch.randn(1,3,IMG_SIZE,IMG_SIZE,device=DEVICE)
    class _W(nn.Module):
        def __init__(s,m): super().__init__(); s.m=m
        def forward(s,z): return s.m(z)["logits"]
    # thop restores the wrapper's train() flag on exit, which would flip the real model back to train mode -> profile a copy
    _copy = copy.deepcopy(model).eval()
    flops,_=thop_profile(_W(_copy).eval(), inputs=(x,), verbose=False)
    del _copy
    model.eval()
    with torch.no_grad():
        for _ in range(warm): model(x)
        torch.cuda.synchronize() if DEVICE=="cuda" else None
        ts=[]
        for _ in range(n_iter):
            t=time.perf_counter(); model(x)
            torch.cuda.synchronize() if DEVICE=="cuda" else None
            ts.append((time.perf_counter()-t)*1000)
    size_mb=sum(p.numel()*4 for p in model.parameters())/1e6
    return {"flops_G":flops/1e9, "latency_ms_mean":float(np.mean(ts)), "latency_ms_sd":float(np.std(ts)), "size_MB":size_mb}

ARMS = {
    "Full (SMGF-Net + MS-CA-ECA)": dict(),
    "Plain backbone (spatial branch only; no morphology, fusion, or attention)": dict(use_spatial=True, use_morphology=False, use_gated_fusion=False, attention="none"),
}
if RUN_REMOVAL_ARMS:
    ARMS.update({
        "- Spatial branch":  dict(use_spatial=False),
        "- Morphology branch": dict(use_morphology=False),
        "- Gated Fusion (50/50 average)": dict(use_gated_fusion=False),
        "- MS-CA-ECA": dict(attention="none"),
    })
RES_PATH = OUT_DIR/"plain_backbone_results.csv"; PRED_PATH = OUT_DIR/"plain_backbone_predictions.npz"
rows=[]; preds={}
for base in [Path(RESUME_DIR) if RESUME_DIR else None, OUT_DIR]:
    if base and (base/"plain_backbone_results.csv").exists() and (base/"plain_backbone_predictions.npz").exists():
        rows=pd.read_csv(base/"plain_backbone_results.csv").to_dict("records"); z=np.load(base/"plain_backbone_predictions.npz", allow_pickle=True); preds={k:z[k] for k in z.files}; break
done={(r["arm"], r["seed"]) for r in rows}

for seed in SEED_LIST:
    for arm, kw in ARMS.items():
        if (arm, seed) in done: print("skip (done):", arm, seed); continue
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        res = run_experiment(f"arm_{abs(hash(arm))%10**6}_s{seed}", BRISC_TRAIN_ROOT, BRISC_TEST_ROOT, BRISC_FOLDER_TO_CLASS,
                             model_builder=build_smgf_mscaeca, model_kwargs=kw)
        m=res["model"]; y,p1,pt=predict_tta(m,res["test_samples"]); tm=res["test_metrics"]
        rows.append({"arm":arm,"seed":seed,"params":res["total_params"],"acc_tta":float((pt.argmax(1)==y).mean()),"acc_single":float((p1.argmax(1)==y).mean()),
                     "f1_macro_tta":tm["f1_mac"],"auc_tta":tm["auc"],"ece_tta":tm["ece"],"n_test":len(y)})
        preds[f"{arm}|{seed}|y"]=y; preds[f"{arm}|{seed}|ptta"]=pt
        pd.DataFrame(rows).to_csv(RES_PATH,index=False); np.savez_compressed(PRED_PATH,**preds)
        print(f"[saved] {arm} seed {seed}: acc_tta={rows[-1]['acc_tta']:.4f}  acc_single={rows[-1]['acc_single']:.4f}  params={res['total_params']:,}")
        del m,res; torch.cuda.empty_cache()


## §3 · Summary and paired tests against the freshly trained full model

In [ ]:
df=pd.DataFrame(rows); print(df.round(4).to_string())
full="Full (SMGF-Net + MS-CA-ECA)"; out=[]
for seed in sorted(df["seed"].unique()):
    if f"{full}|{seed}|y" not in preds: continue
    y=preds[f"{full}|{seed}|y"]; ca=preds[f"{full}|{seed}|ptta"].argmax(1)==y
    for arm in df["arm"].unique():
        if arm==full or f"{arm}|{seed}|y" not in preds: continue
        cb=preds[f"{arm}|{seed}|ptta"].argmax(1)==y
        b,cc,p=mcnemar_exact(ca,cb); d,lo,hi=paired_bootstrap_diff(ca,cb)
        out.append({"seed":seed,"arm":arm,"acc_full":ca.mean(),"acc_arm":cb.mean(),"full_right_arm_wrong":b,"full_wrong_arm_right":cc,"mcnemar_p":p,"acc_diff_full_minus_arm":d,"ci_lo":lo,"ci_hi":hi})
mc=pd.DataFrame(out); mc.to_csv(OUT_DIR/"plain_backbone_mcnemar.csv",index=False); print(mc.round(4).to_string())
fs=df[df["arm"]==full]["acc_tta"].values
if len(fs)>1: print(f"\nFull model across seeds: mean {fs.mean():.4f}, SD {fs.std(ddof=1):.4f}")
print("\nFor the paper: report the plain-backbone row next to the full model from THIS run, and quote the full-model accuracy of this run (not 96.28 from the earlier run) when computing deltas.")
